# Preprocessing Finders Utilities Demo

Demostración interactiva de las funciones de búsqueda y localización de anomalías estructurales (duplicados de tiempo, discontinuidades temporales y correspondencia de columnas derivadas) implementadas en `_preprocessing_finders.py` y expuestas por `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `find_duplicates`
- `find_large_gaps`
- `find_column_pairs`

## Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    find_duplicates,
    find_large_gaps,
    find_column_pairs,
)

## find_duplicates

`find_duplicates` identifica marcas de tiempo duplicadas y devuelve tanto el número de timestamps únicos que están repetidos como un DataFrame con todas las filas involucradas (incluyendo la primera ocurrencia).

In [2]:
# 1. Datos con timestamps repetidos
df_dups = pd.DataFrame({
    "timestamp": [
        "2024-01-01 12:00:00",
        "2024-01-01 12:00:00",  # repetido
        "2024-01-01 12:01:00",
        "2024-01-01 12:02:00",
        "2024-01-01 12:02:00",  # repetido
        "2024-01-01 12:03:00",
    ],
    "reading": [10.2, 10.5, 11.0, 15.3, 15.4, 12.8]
})

# 2. Detección
n_unique_dups, df_all_dups = find_duplicates(df=df_dups, col_time="timestamp")

print(f"Número de timestamps únicos repetidos: {n_unique_dups}")
print("Filas completas afectadas por duplicidad:")
display(df_all_dups)

Número de timestamps únicos repetidos: 2
Filas completas afectadas por duplicidad:


,timestamp,reading
0,2024-01-01 12:00:00,10.2
1,2024-01-01 12:00:00,10.5
3,2024-01-01 12:02:00,15.3
4,2024-01-01 12:02:00,15.4


## find_large_gaps

`find_large_gaps` localiza interrupciones o saltos temporales que superan la tolerancia esperada (`freq * limit`). Devuelve un DataFrame con el timestamp inicial y final del hueco, el número de periodos perdidos y la duración total.

In [3]:
# 1. Serie temporal con un hueco de 2h y un hueco de 5h
df_gaps = pd.DataFrame({
    "recorded_at": [
        "2024-01-01 00:00:00",
        "2024-01-01 01:00:00",
        "2024-01-01 02:00:00",
        # Hueco de 2 horas (1 periodo ausente)
        "2024-01-01 04:00:00",
        "2024-01-01 05:00:00",
        # Hueco de 5 horas (4 periodos ausentes)
        "2024-01-01 10:00:00",
    ],
    "metric": [10, 11, 12, 14, 15, 20]
})

# 2. Detección con frecuencia 1h y límite 1 período
df_detected_gaps = find_large_gaps(
    df=df_gaps,
    col_time="recorded_at",
    freq="1h",
    limit=1
)

print("Gaps temporales no tolerables detectados:")
display(df_detected_gaps)

Gaps temporales no tolerables detectados:


,start,end,missing_periods,duration
0,2024-01-01 02:00:00,2024-01-01 04:00:00,1,0 days 02:00:00
1,2024-01-01 05:00:00,2024-01-01 10:00:00,4,0 days 05:00:00


## find_column_pairs

`find_column_pairs` encuentra automáticamente las correspondencias entre columnas originales y sus versiones procesadas o sustitutas mediante un prefijo o sufijo naming convention.

In [4]:
# 1. DataFrame con pares de columnas originales y procesadas
df_pairs = pd.DataFrame({
    "temperature": [20.1, 20.2],
    "temperature_clean": [20.1, 20.2],
    "humidity": [60, 65],
    "humidity_clean": [60, 65],
    "clean_pressure": [1013, 1014],
    "pressure": [1013, 1014],
    "status": ["OK", "OK"]
})

# 2. Búsqueda por sufijo '_clean'
orig_cols, repl_cols = find_column_pairs(df=df_pairs, suffix="_clean")

print("Columnas originales identificadas:", orig_cols)
print("Columnas de sustitución asociadas:", repl_cols)

Columnas originales identificadas: ['humidity', 'temperature']
Columnas de sustitución asociadas: ['humidity_clean', 'temperature_clean']
